Goal : Find out what cell types/cell states express PYCR1, CD5L, ANGPT4, ETV4, and ADM2.

In [1]:
import GEOparse

gse_sc = GEOparse.get_GEO(
    geo="GSE131907",
    destdir="../data/raw/geo",
)

len(gse_sc.gsms), list(gse_sc.gsms.keys())[:5]

01-Jul-2026 21:16:12 DEBUG utils - Directory ../data/raw/geo already exists. Skipping.
01-Jul-2026 21:16:12 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/soft/GSE131907_family.soft.gz to ../data/raw/geo/GSE131907_family.soft.gz
100%|██████████| 6.92k/6.92k [00:01<00:00, 5.44kB/s]
01-Jul-2026 21:16:15 DEBUG downloader - Size validation passed
01-Jul-2026 21:16:15 DEBUG downloader - Moving /var/folders/48/j7bwkjs14079sd8gcjdn18cw0000gn/T/tmpi6zu1f4u to /Users/adityaanand/dev/cancer-lab/data/raw/geo/GSE131907_family.soft.gz
01-Jul-2026 21:16:15 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/soft/GSE131907_family.soft.gz
01-Jul-2026 21:16:15 INFO GEOparse - Parsing ../data/raw/geo/GSE131907_family.soft.gz: 
01-Jul-2026 21:16:15 DEBUG GEOparse - DATABASE: GeoMiame
01-Jul-2026 21:16:15 DEBUG GEOparse - SERIES: GSE131907
01-Jul-2026 21:16:15 DEBUG GEOparse - PLATFORM: GPL16791
01-Jul-2026 21:16:15 DE

(58, ['GSM3827114', 'GSM3827115', 'GSM3827116', 'GSM3827117', 'GSM3827118'])

In [2]:
gse_sc.metadata.keys()

dict_keys(['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'pubmed_id', 'web_link', 'summary', 'overall_design', 'type', 'contributor', 'sample_id', 'contact_name', 'contact_email', 'contact_phone', 'contact_department', 'contact_institute', 'contact_address', 'contact_city', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'platform_id', 'platform_taxid', 'sample_taxid', 'relation'])

In [3]:
gse_sc.metadata.get("supplementary_file", [])

['ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/suppl/GSE131907_Lung_Cancer_Feature_Summary.xlsx',
 'ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/suppl/GSE131907_Lung_Cancer_cell_annotation.txt.gz',
 'ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/suppl/GSE131907_Lung_Cancer_normalized_log2TPM_matrix.rds.gz',
 'ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/suppl/GSE131907_Lung_Cancer_normalized_log2TPM_matrix.txt.gz',
 'ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/suppl/GSE131907_Lung_Cancer_raw_UMI_matrix.rds.gz',
 'ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/suppl/GSE131907_Lung_Cancer_raw_UMI_matrix.txt.gz']

In [4]:
for gsm_name, gsm in list(gse_sc.gsms.items())[:3]:
    print(gsm_name)
    print(gsm.metadata.keys())
    print(gsm.metadata)
    print()

GSM3827114
dict_keys(['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1', 'growth_protocol_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_phone', 'contact_department', 'contact_institute', 'contact_address', 'contact_city', 'contact_zip/postal_code', 'contact_country', 'instrument_model', 'library_selection', 'library_source', 'library_strategy', 'relation', 'supplementary_file_1', 'series_id', 'data_row_count'])
{'title': ['LUNG_N01'], 'geo_accession': ['GSM3827114'], 'status': ['Public on Mar 27 2020'], 'submission_date': ['May 29 2019'], 'last_update_date': ['Aug 02 2022'], 'type': ['SRA'], 'channel_count': ['1'], 'source_name_ch1': ['Normal lung'], 'organism_ch1': ['Homo sapiens'], 'taxid_ch1': ['9606'], 'characteristics_ch1': ['patient id: P0001', 'tumor stage: I', 'tissue or

In [5]:
import subprocess
from pathlib import Path

sc_dir = Path("../data/raw/geo/GSE131907")
sc_dir.mkdir(parents=True, exist_ok=True)

# expected Content-Length from GEO (bytes)
files = {
    "annotation": (
        "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/suppl/GSE131907_Lung_Cancer_cell_annotation.txt.gz",
        None,
    ),
    "normalized": (
        "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE131nnn/GSE131907/suppl/GSE131907_Lung_Cancer_normalized_log2TPM_matrix.txt.gz",
        3_065_932_358,
    ),
}


def gzip_is_valid(path: Path) -> bool:
    try:
        subprocess.run(["gzip", "-t", str(path)], check=True, capture_output=True)
        return True
    except subprocess.CalledProcessError:
        return False


def download_geo_file(url: str, out_path: Path, expected_bytes: int | None = None) -> Path:
    if out_path.exists():
        size_ok = expected_bytes is None or out_path.stat().st_size == expected_bytes
        if size_ok and gzip_is_valid(out_path):
            print("Already exists:", out_path)
            return out_path
        print("Removing corrupt/incomplete file:", out_path)
        out_path.unlink()

    tmp_path = out_path.with_suffix(out_path.suffix + ".part")
    print("Downloading", out_path.name)
    subprocess.run(["curl", "-L", "-o", str(tmp_path), url], check=True)

    if expected_bytes is not None and tmp_path.stat().st_size != expected_bytes:
        tmp_path.unlink()
        raise RuntimeError(
            f"Download size mismatch for {out_path.name}: "
            f"got {tmp_path.stat().st_size:,}, expected {expected_bytes:,}"
        )

    tmp_path.rename(out_path)
    if not gzip_is_valid(out_path):
        out_path.unlink()
        raise RuntimeError(f"Downloaded file failed gzip integrity check: {out_path}")

    print("Downloaded and verified:", out_path)
    return out_path


for name, (url, expected_bytes) in files.items():
    out_path = sc_dir / url.split("/")[-1]
    download_geo_file(url, out_path, expected_bytes)

In [6]:
import pandas as pd
import gzip

annot_path = sc_dir / "GSE131907_Lung_Cancer_cell_annotation.txt.gz"

cell_annot = pd.read_csv(
    annot_path,
    sep="\t",
    compression="gzip",
)

cell_annot.shape, cell_annot.head(), cell_annot.columns.tolist()

((208506, 7),
                        Index           Barcode    Sample Sample_Origin  \
 0  AAACCTGCAAGGTGTG_LUNG_N01  AAACCTGCAAGGTGTG  LUNG_N01         nLung   
 1  AACTCCCGTTCACCTC_LUNG_N01  AACTCCCGTTCACCTC  LUNG_N01         nLung   
 2  AACTCCCTCACGCGGT_LUNG_N01  AACTCCCTCACGCGGT  LUNG_N01         nLung   
 3  AAGGAGCGTGTGCGTC_LUNG_N01  AAGGAGCGTGTGCGTC  LUNG_N01         nLung   
 4  AAGGTTCAGGTACTCT_LUNG_N01  AAGGTTCAGGTACTCT  LUNG_N01         nLung   
 
        Cell_type Cell_type.refined Cell_subtype  
 0  Myeloid cells     Myeloid cells       mo-Mac  
 1  Myeloid cells     Myeloid cells       mo-Mac  
 2  Myeloid cells     Myeloid cells       mo-Mac  
 3  Myeloid cells     Myeloid cells       mo-Mac  
 4  Myeloid cells     Myeloid cells       mo-Mac  ,
 ['Index',
  'Barcode',
  'Sample',
  'Sample_Origin',
  'Cell_type',
  'Cell_type.refined',
  'Cell_subtype'])

In [7]:
matrix_path = sc_dir / "GSE131907_Lung_Cancer_normalized_log2TPM_matrix.txt.gz"

with gzip.open(matrix_path, "rt") as f:
    for _ in range(3):
        line = f.readline()
        print(line[:500])

Index	AAACCTGAGAAACCGC_LN_05	AAACCTGAGAAACGCC_NS_13	AAACCTGAGAAGGTGA_LUNG_N18	AAACCTGAGACAAAGG_LUNG_N18	AAACCTGAGACATAAC_LN_04	AAACCTGAGACCTTTG_LUNG_N30	AAACCTGAGACGACGT_LUNG_T09	AAACCTGAGACGACGT_LUNG_T34	AAACCTGAGACGCTTT_LUNG_T18	AAACCTGAGACGCTTT_NS_04	AAACCTGAGACTGGGT_LUNG_N30	AAACCTGAGACTGTAA_LUNG_T34	AAACCTGAGAGCAATT_EFFUSION_06	AAACCTGAGAGCAATT_LUNG_N31	AAACCTGAGAGCCTAG_LUNG_N20	AAACCTGAGAGGGATA_LUNG_N08	AAACCTGAGAGGGATA_NS_19	AAACCTGAGAGGTTGC_BRONCHO_11	AAACCTGAGAGTGACC_BRONCHO_58	AAACCTGA
A1BG	0	0	0	0	0	0	0	0	0	0	0	0	0	1.56691670534781	0	0	1.28828593084874	0	0	0	0	2.46203185071558	0	0	0	1.1839253585351	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	2.00336077899624	2.55125728425497	0	0	0	2.12664364345093	0	0	0	0	0	0	0	0	0	0	0	1.3165200963317	0	0	1.5579689890816	0	0	0	0	0	0	0	0	2.03894628024052	0	0	0	0	0	0	0	0	0	0	0	3.4360529014894	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	1.44753373898505	2.64243321940823	0	0	0	0	0	0	0	0	0	0.421857031598742	2.57063571922205	0	0	0	0	0	0	

In [9]:
import subprocess

target_genes = ["PYCR1", "CD5L", "ANGPT4", "ETV4", "ADM2"]
target_gene_set = set(target_genes)
gene_pattern = "^(" + "|".join(target_genes) + ")\t"

with gzip.open(matrix_path, "rt") as f:
    header = f.readline().rstrip("\n")

gene_lines = subprocess.run(
    ["zgrep", "-E", gene_pattern, str(matrix_path)],
    capture_output=True,
    text=True,
    check=True,
).stdout.rstrip("\n").split("\n")

selected_lines = [header] + [line for line in gene_lines if line]
found_genes = {line.split("\t", 1)[0] for line in gene_lines if line}
missing_genes = target_gene_set - found_genes

len(selected_lines), sorted(found_genes), sorted(missing_genes)

(6, ['ADM2', 'ANGPT4', 'CD5L', 'ETV4', 'PYCR1'], [])

In [10]:
small_matrix_path = sc_dir / "GSE131907_selected_gene_log2TPM.tsv"

with open(small_matrix_path, "w") as f:
    for line in selected_lines:
        f.write(line + "\n")

small_matrix_path

PosixPath('../data/raw/geo/GSE131907/GSE131907_selected_gene_log2TPM.tsv')

In [11]:
selected_expr = pd.read_csv(
    small_matrix_path,
    sep="\t",
    index_col=0,
)

selected_expr.shape, selected_expr.head()

((5, 208506),
         AAACCTGAGAAACCGC_LN_05  AAACCTGAGAAACGCC_NS_13  \
 Index                                                    
 ADM2                         0                       0   
 ANGPT4                       0                       0   
 CD5L                         0                       0   
 ETV4                         0                       0   
 PYCR1                        0                       0   
 
         AAACCTGAGAAGGTGA_LUNG_N18  AAACCTGAGACAAAGG_LUNG_N18  \
 Index                                                          
 ADM2                            0                          0   
 ANGPT4                          0                          0   
 CD5L                            0                          0   
 ETV4                            0                          0   
 PYCR1                           0                          0   
 
         AAACCTGAGACATAAC_LN_04  AAACCTGAGACCTTTG_LUNG_N30  \
 Index                                              

In [12]:
selected_expr_cells = selected_expr.T

selected_expr_cells.shape, selected_expr_cells.head()

((208506, 5),
 Index                      ADM2  ANGPT4  CD5L  ETV4  PYCR1
 AAACCTGAGAAACCGC_LN_05      0.0     0.0   0.0   0.0    0.0
 AAACCTGAGAAACGCC_NS_13      0.0     0.0   0.0   0.0    0.0
 AAACCTGAGAAGGTGA_LUNG_N18   0.0     0.0   0.0   0.0    0.0
 AAACCTGAGACAAAGG_LUNG_N18   0.0     0.0   0.0   0.0    0.0
 AAACCTGAGACATAAC_LN_04      0.0     0.0   0.0   0.0    0.0)

In [13]:
selected_expr_cells.index[:5], cell_annot.head()

(Index(['AAACCTGAGAAACCGC_LN_05', 'AAACCTGAGAAACGCC_NS_13',
        'AAACCTGAGAAGGTGA_LUNG_N18', 'AAACCTGAGACAAAGG_LUNG_N18',
        'AAACCTGAGACATAAC_LN_04'],
       dtype='str'),
                        Index           Barcode    Sample Sample_Origin  \
 0  AAACCTGCAAGGTGTG_LUNG_N01  AAACCTGCAAGGTGTG  LUNG_N01         nLung   
 1  AACTCCCGTTCACCTC_LUNG_N01  AACTCCCGTTCACCTC  LUNG_N01         nLung   
 2  AACTCCCTCACGCGGT_LUNG_N01  AACTCCCTCACGCGGT  LUNG_N01         nLung   
 3  AAGGAGCGTGTGCGTC_LUNG_N01  AAGGAGCGTGTGCGTC  LUNG_N01         nLung   
 4  AAGGTTCAGGTACTCT_LUNG_N01  AAGGTTCAGGTACTCT  LUNG_N01         nLung   
 
        Cell_type Cell_type.refined Cell_subtype  
 0  Myeloid cells     Myeloid cells       mo-Mac  
 1  Myeloid cells     Myeloid cells       mo-Mac  
 2  Myeloid cells     Myeloid cells       mo-Mac  
 3  Myeloid cells     Myeloid cells       mo-Mac  
 4  Myeloid cells     Myeloid cells       mo-Mac  )

In [14]:
cell_annot.columns.tolist()

['Index',
 'Barcode',
 'Sample',
 'Sample_Origin',
 'Cell_type',
 'Cell_type.refined',
 'Cell_subtype']

In [19]:
# alignment uses Index = "{barcode}_{sample}" (same as matrix column names)

In [18]:
# matrix columns and annotation rows both use "{barcode}_{sample}"
cell_annot_indexed = cell_annot.set_index("Index")

common_cells = selected_expr_cells.index.intersection(cell_annot_indexed.index)

len(common_cells), selected_expr_cells.shape[0], cell_annot_indexed.shape[0]

(208506, 208506, 208506)

In [20]:
expr_sc = selected_expr_cells.loc[common_cells].copy()
annot_sc = cell_annot_indexed.loc[common_cells].copy()

expr_sc.shape, annot_sc.shape

((208506, 5), (208506, 6))

In [21]:
annot_sc.columns.tolist(), annot_sc.head()

(['Barcode',
  'Sample',
  'Sample_Origin',
  'Cell_type',
  'Cell_type.refined',
  'Cell_subtype'],
                                     Barcode    Sample Sample_Origin  \
 AAACCTGAGAAACCGC_LN_05     AAACCTGAGAAACCGC     LN_05           nLN   
 AAACCTGAGAAACGCC_NS_13     AAACCTGAGAAACGCC     NS_13        mBrain   
 AAACCTGAGAAGGTGA_LUNG_N18  AAACCTGAGAAGGTGA  LUNG_N18         nLung   
 AAACCTGAGACAAAGG_LUNG_N18  AAACCTGAGACAAAGG  LUNG_N18         nLung   
 AAACCTGAGACATAAC_LN_04     AAACCTGAGACATAAC     LN_04           nLN   
 
                                   Cell_type Cell_type.refined     Cell_subtype  
 AAACCTGAGAAACCGC_LN_05        B lymphocytes     B lymphocytes     MALT B cells  
 AAACCTGAGAAACGCC_NS_13     Epithelial cells  Epithelial cells  Malignant cells  
 AAACCTGAGAAGGTGA_LUNG_N18     T lymphocytes        T/NK cells          CD4+ Th  
 AAACCTGAGACAAAGG_LUNG_N18     Myeloid cells     Myeloid cells        Monocytes  
 AAACCTGAGACATAAC_LN_04        T lymphocytes        T/N

In [22]:
celltype_col = "Cell_type"
sample_col = "Sample_Origin"

In [23]:
audit_df = expr_sc.copy()
audit_df["cell_type"] = annot_sc[celltype_col].values

mean_by_celltype = (
    audit_df
    .groupby("cell_type")[target_genes]
    .mean()
)

pct_by_celltype = (
    audit_df
    .groupby("cell_type")[target_genes]
    .apply(lambda x: (x > 0).mean())
)

mean_by_celltype

Index,PYCR1,CD5L,ANGPT4,ETV4,ADM2
cell_type,,,,,
B lymphocytes,0.039526,0.000844,0.000165,0.001803,0.006566
Endothelial cells,0.027272,0.003063,0.000809,0.008165,0.000332
Epithelial cells,0.584246,0.000790,0.000337,0.159200,0.047184
Fibroblasts,0.275579,0.001147,0.002681,0.025359,0.011589
MAST cells,0.015569,0.000917,0.000953,0.002081,0.000478
Myeloid cells,0.014150,0.048592,0.016649,0.002746,0.002216
NK cells,0.005956,0.002123,0.000739,0.001265,0.004625
Oligodendrocytes,0.063702,0.000000,0.000000,0.011202,0.016717
T lymphocytes,0.009028,0.000470,0.000272,0.001816,0.004465


In [24]:
pct_by_celltype

Index,PYCR1,CD5L,ANGPT4,ETV4,ADM2
cell_type,,,,,
B lymphocytes,0.039809,0.000506,0.000145,0.001482,0.006038
Endothelial cells,0.023547,0.002505,0.001503,0.005511,0.000501
Epithelial cells,0.437437,0.000850,0.000686,0.180053,0.069268
Fibroblasts,0.209012,0.001438,0.001678,0.022052,0.010786
MAST cells,0.008539,0.000883,0.000883,0.000883,0.000294
Myeloid cells,0.012759,0.058824,0.022133,0.002201,0.002107
NK cells,0.002597,0.000866,0.000346,0.000519,0.001991
Oligodendrocytes,0.048883,0.000000,0.000000,0.006983,0.020950
T lymphocytes,0.005096,0.000226,0.000126,0.000816,0.002234


In [25]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

mean_by_celltype.to_csv(processed_dir / "gse131907_selected_gene_mean_by_celltype.csv")
pct_by_celltype.to_csv(processed_dir / "gse131907_selected_gene_pct_by_celltype.csv")

In [26]:
mean_long = (
    mean_by_celltype
    .reset_index()
    .melt(
        id_vars="cell_type",
        var_name="gene",
        value_name="mean_log2TPM",
    )
)

pct_long = (
    pct_by_celltype
    .reset_index()
    .melt(
        id_vars="cell_type",
        var_name="gene",
        value_name="fraction_expressing",
    )
)

sc_audit_long = mean_long.merge(
    pct_long,
    on=["cell_type", "gene"],
    how="left",
)

sc_audit_long = sc_audit_long.sort_values(
    ["gene", "mean_log2TPM"],
    ascending=[True, False],
)

sc_audit_long

,cell_type,gene,mean_log2TPM,fraction_expressing
42,Epithelial cells,ADM2,0.047184,0.069268
47,Oligodendrocytes,ADM2,0.016717,0.020950
43,Fibroblasts,ADM2,0.011589,0.010786
49,Undetermined,ADM2,0.009094,0.006349
40,B lymphocytes,ADM2,0.006566,0.006038
46,NK cells,ADM2,0.004625,0.001991
48,T lymphocytes,ADM2,0.004465,0.002234
45,Myeloid cells,ADM2,0.002216,0.002107
44,MAST cells,ADM2,0.000478,0.000294
41,Endothelial cells,ADM2,0.000332,0.000501


In [27]:
sc_audit_long.to_csv(
    processed_dir / "gse131907_selected_gene_single_cell_audit_long.csv",
    index=False,
)

In [28]:
from IPython.display import display

for gene in target_genes:
    print("\n", gene)
    display(
        sc_audit_long[sc_audit_long["gene"] == gene]
        .sort_values("mean_log2TPM", ascending=False)
        .head(10)
    )


 PYCR1


,cell_type,gene,mean_log2TPM,fraction_expressing
2,Epithelial cells,PYCR1,0.584246,0.437437
3,Fibroblasts,PYCR1,0.275579,0.209012
7,Oligodendrocytes,PYCR1,0.063702,0.048883
0,B lymphocytes,PYCR1,0.039526,0.039809
1,Endothelial cells,PYCR1,0.027272,0.023547
9,Undetermined,PYCR1,0.025098,0.017460
4,MAST cells,PYCR1,0.015569,0.008539
5,Myeloid cells,PYCR1,0.014150,0.012759
8,T lymphocytes,PYCR1,0.009028,0.005096
6,NK cells,PYCR1,0.005956,0.002597



 CD5L


,cell_type,gene,mean_log2TPM,fraction_expressing
15,Myeloid cells,CD5L,0.048592,0.058824
11,Endothelial cells,CD5L,0.003063,0.002505
16,NK cells,CD5L,0.002123,0.000866
13,Fibroblasts,CD5L,0.001147,0.001438
14,MAST cells,CD5L,0.000917,0.000883
10,B lymphocytes,CD5L,0.000844,0.000506
12,Epithelial cells,CD5L,0.000790,0.000850
18,T lymphocytes,CD5L,0.000470,0.000226
17,Oligodendrocytes,CD5L,0.000000,0.000000
19,Undetermined,CD5L,0.000000,0.000000



 ANGPT4


,cell_type,gene,mean_log2TPM,fraction_expressing
25,Myeloid cells,ANGPT4,0.016649,0.022133
23,Fibroblasts,ANGPT4,0.002681,0.001678
24,MAST cells,ANGPT4,0.000953,0.000883
21,Endothelial cells,ANGPT4,0.000809,0.001503
26,NK cells,ANGPT4,0.000739,0.000346
22,Epithelial cells,ANGPT4,0.000337,0.000686
28,T lymphocytes,ANGPT4,0.000272,0.000126
20,B lymphocytes,ANGPT4,0.000165,0.000145
27,Oligodendrocytes,ANGPT4,0.000000,0.000000
29,Undetermined,ANGPT4,0.000000,0.000000



 ETV4


,cell_type,gene,mean_log2TPM,fraction_expressing
32,Epithelial cells,ETV4,0.159200,0.180053
33,Fibroblasts,ETV4,0.025359,0.022052
37,Oligodendrocytes,ETV4,0.011202,0.006983
31,Endothelial cells,ETV4,0.008165,0.005511
35,Myeloid cells,ETV4,0.002746,0.002201
34,MAST cells,ETV4,0.002081,0.000883
38,T lymphocytes,ETV4,0.001816,0.000816
30,B lymphocytes,ETV4,0.001803,0.001482
36,NK cells,ETV4,0.001265,0.000519
39,Undetermined,ETV4,0.000000,0.000000



 ADM2


,cell_type,gene,mean_log2TPM,fraction_expressing
42,Epithelial cells,ADM2,0.047184,0.069268
47,Oligodendrocytes,ADM2,0.016717,0.020950
43,Fibroblasts,ADM2,0.011589,0.010786
49,Undetermined,ADM2,0.009094,0.006349
40,B lymphocytes,ADM2,0.006566,0.006038
46,NK cells,ADM2,0.004625,0.001991
48,T lymphocytes,ADM2,0.004465,0.002234
45,Myeloid cells,ADM2,0.002216,0.002107
44,MAST cells,ADM2,0.000478,0.000294
41,Endothelial cells,ADM2,0.000332,0.000501
